In [13]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model=ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [4]:
class EvaluationSchema(BaseModel):
    
    feedback:str=Field(description="Detailed feedback on the essay")
    score:int=Field(description="Score out of 10",ge=0,le=10)
    

In [6]:
structured_model=model.with_structured_output(EvaluationSchema)

In [10]:
essay="""
The Indian economy has been a subject of extensive analysis and discussion, particularly in the context of its growth trajectory, structural challenges, and policy interventions. Over the past few decades, India has witnessed significant economic reforms that have propelled it towards becoming one of the fastest-growing major economies in the world. However, despite these advancements, the country continues to grapple with issues such as income inequality, unemployment, and infrastructural deficits. The economic landscape of India is characterized by a diverse mix of agriculture, manufacturing, and services sectors,
each contributing to the overall GDP in varying degrees. The government's role in shaping economic policies, promoting foreign investment, and fostering innovation has been pivotal in driving growth. Additionally, the impact of globalization and technological advancements has further influenced the dynamics of the Indian economy, presenting both opportunities and challenges for sustainable development.
In conclusion, while the Indian economy has made remarkable strides in recent years, it is imperative to address the underlying structural issues and ensure inclusive growth to achieve long-term economic stability and prosperity."""

prompt=f"Evaluate the following essay and provide feedback and a score out of 10:\n\n{essay}"

structured_model.invoke(prompt).score

8

In [14]:
class UPSCState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int], operator.add]
    avg_score:float
    

In [20]:
def evaluate_language(state:UPSCState)->UPSCState:
    prompt=f"Evaluate the language and grammar of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    
    result=structured_model.invoke(prompt)
    return {'language_feedback':result.feedback,"individual_scores":[result.score]}

In [19]:
def evaluate_analysis(state:UPSCState)->UPSCState:
    prompt=f"Evaluate the depth of analysis of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    
    result=structured_model.invoke(prompt)
    return {'analysis_feedback':result.feedback,"individual_scores":[result.score]}
    

In [18]:
def evaluate_thought(state:UPSCState)->UPSCState:
    prompt=f"Evaluate the clarity of thought of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    
    result=structured_model.invoke(prompt)
    return {'clarity_feedback':result.feedback,"individual_scores":[result.score]}

In [21]:
def final_evaluation(state:UPSCState)->UPSCState:
    prompt=f"Based on the following feedbacks create a summarized feedback \n language feedback: {state['language_feedback']}\n analysis feedback: {state['analysis_feedback']}\n clarity feedback: {state['clarity_feedback']}\n\ngive an overall score out of 10:\n\n{state['essay']}"
    
    result=model.invoke(prompt).content
    
    overall_score=sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback':result,"avg_score":overall_score}

In [24]:
graph=StateGraph(UPSCState)

#Nodes for language, analysis and clarity feedback
graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_analysis",evaluate_analysis)
graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("final_evaluation",final_evaluation)

#Edges 
graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_analysis")
graph.add_edge(START,"evaluate_thought")

graph.add_edge("evaluate_language","final_evaluation")
graph.add_edge("evaluate_analysis","final_evaluation")
graph.add_edge("evaluate_thought","final_evaluation")

graph.add_edge("final_evaluation",END)

workflows=graph.compile()


In [26]:
intial_state={
    'essay':essay,
}
result=workflows.invoke(intial_state)
print(result)


{'essay': "\nThe Indian economy has been a subject of extensive analysis and discussion, particularly in the context of its growth trajectory, structural challenges, and policy interventions. Over the past few decades, India has witnessed significant economic reforms that have propelled it towards becoming one of the fastest-growing major economies in the world. However, despite these advancements, the country continues to grapple with issues such as income inequality, unemployment, and infrastructural deficits. The economic landscape of India is characterized by a diverse mix of agriculture, manufacturing, and services sectors,\neach contributing to the overall GDP in varying degrees. The government's role in shaping economic policies, promoting foreign investment, and fostering innovation has been pivotal in driving growth. Additionally, the impact of globalization and technological advancements has further influenced the dynamics of the Indian economy, presenting both opportunities 